In [34]:
from pathlib import Path
import pandas as pd
import torch.nn as nn
import torch
from torch.utils.data import DataLoader
from ombs_senegal.time_series_deepl import Learner, HydroDataset, split_by_date
from sklearn.preprocessing import RobustScaler


DATA_PATH = Path("../../data")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Data preprocessing

In [35]:
data = pd.read_csv(
    DATA_PATH/'data_cumul.csv', 
    sep=';', 
    usecols=['time', 'débit_insitu', 'P_cumul_7j', 'débit_mgb'], 
    index_col='time',
    converters={"time": pd.to_datetime}
    )
data = data["2012-01-01":]
data["mois"] = data.index.month

In [36]:
train, valid, test = split_by_date(data, val_dates=("2018-01-01", "2018-12-31"), test_dates=("2019-01-01", "2020-12-31"))

Approximate data repartition:
Train: 66.67%
Validation: 11.10%
Test: 22.23%


Now lets define the feature and the target columns and divide data in feature and targets

In [37]:
from ombs_senegal.season import SeasonalityHandler

In [38]:
x_cols = ["débit_mgb","P_cumul_7j", "mois"]
y_cols = ["débit_insitu"]

x_train, y_train = train[x_cols], train[y_cols]
x_valid, y_valid = valid[x_cols], valid[y_cols]
x_test, y_test = test[x_cols], test[y_cols]

In [39]:
season_handler =SeasonalityHandler()
_ = season_handler.compute_seasonal_pattern(y_train)
y_train = season_handler.remove_seasonality(y_train)
y_valid = season_handler.remove_seasonality(y_valid)


In [40]:
feature_scaler, target_scaler = RobustScaler(), RobustScaler()
_, _ = feature_scaler.fit_transform(x_train), target_scaler.fit_transform(y_train)

## Model definition

#### Multi layer perceptron (MLP)

In [41]:
class SimpleRegularizedMLP(nn.Module):
    def __init__(self, input_size, prediction_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.norm = nn.LayerNorm(64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(64, prediction_size)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.norm(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


## Trainning

In [42]:
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from ombs_senegal.benchmark_model import BenchmarkScores

In [45]:

# 🔹 Listes des tailles de fenêtres à tester
context_sizes = [15, 30, 60, 90]
batch_size = 32
learning_rate = 0.0003
epochs=15

prediction_size = 10  # Fixe (peut être ajusté)
x_transform=feature_scaler.transform
y_transform=target_scaler.transform
results = []
models = []
benchmark_scores = BenchmarkScores()

# 🔹 Boucle sur différentes tailles de fenêtres
for context_size in context_sizes:
    print(f"\n🟢 Test avec window_size = {context_size}")

    train_dataset = HydroDataset(x=x_train, y=y_train, ctx_len=context_size, pred_len=prediction_size, x_transform=x_transform, y_transform=y_transform)
    valid_dataset = HydroDataset(x=x_valid, y=y_valid, ctx_len=context_size, pred_len=prediction_size, x_transform=x_transform, y_transform=y_transform)
    test_dataset = HydroDataset(x=x_test, y=y_test, ctx_len=context_size, pred_len=prediction_size, x_transform=x_transform, y_transform=y_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    # 🔹 Vérification des dimensions
    # model = LSTMModel(len(x_cols), prediction_size).to(DEVICE)
    #model = SimpleRegularizedLSTM(len(x_cols), 64, prediction_size).to(DEVICE)
    # model = SimpleRegularizedGRU(len(x_cols), 64, prediction_size).to(DEVICE)
    # model = TemporalCausalCNN(len(x_cols), context_size, prediction_size).to(DEVICE)
    model = SimpleRegularizedMLP(len(x_cols)*context_size, prediction_size).to(DEVICE)
    learner = Learner(model=model, train_loader=train_loader, val_loader=valid_loader)
    learner.fit(lr=learning_rate, epochs=epochs)

    y_pred = learner.predict(test_loader, inverse_transform=target_scaler.inverse_transform)

    y_pred.index.name = "time"
    y_pred = season_handler.add_seasonality(y_pred)
    y_pred["model"] = model.__class__.__name__
    y_pred.set_index(["model"], append=True, inplace=True)
    y_pred = y_pred.to_xarray()
    scores = benchmark_scores.compute_scores(y_pred, y_test.to_xarray()[y_cols[0]], metrics=["mae", "rmse", "nse", "kge"])

    mean_scores = {s.upper(): round(float(scores[s].mean().values), 2) for s in scores.data_vars}
    print(f"📊 Résultats pour window_size={context_size} -> {mean_scores}")

    # 🔹 Stocker les résultats
    results.append({"ctx_size": context_size, **mean_scores})

# 🔹 Afficher le meilleur résultat
best = min(results, key=lambda x: x["RMSE"])  # Choix basé sur le RMSE le plus bas
print(f"\n✅ Meilleure fenêtre : {best["ctx_size"]} avec RMSE={best["RMSE"]}, MAE={best["MAE"]}, NSE={best["NSE"]}, KGE={best["KGE"]}")



🟢 Test avec window_size = 15


Training epochs:   0%|          | 0/15 [00:00<?, ?it/s]

Training epochs:   7%|▋         | 1/15 [00:00<00:10,  1.36it/s]

Epoch 1, Loss: 16.8719, Val Loss: 36.9478


Training epochs:  13%|█▎        | 2/15 [00:01<00:08,  1.60it/s]

...

Training epochs:  93%|█████████▎| 14/15 [00:07<00:00,  1.80it/s]

Epoch 14, Loss: 13.8987, Val Loss: 42.5384


Training epochs: 100%|██████████| 15/15 [00:08<00:00,  1.83it/s]

Epoch 15, Loss: 13.6231, Val Loss: 42.0716


📊 Résultats pour window_size=90 -> {'MAE': 81.36, 'RMSE': 162.7, 'NSE': 0.86, 'KGE': 0.84}

✅ Meilleure fenêtre : 15 avec RMSE=148.13, MAE=70.07, NSE=0.87, KGE=0.83


- MLP: Meilleure fenêtre : 60 avec RMSE=156.864, MAE=78.964, R²=0.861
- CNN: Meilleure fenêtre : 30 avec RMSE=191.998, MAE=92.670, R²=0.786
- GRU: Meilleure fenêtre : 60 avec RMSE=212.033, MAE=108.959, R²=0.746
- Simple LSTM: Meilleure fenêtre : 90 avec RMSE=197.514, MAE=104.173, R²=0.785

In [ ]:
# 🔹 Fonction pour calculer le PBIAS
def pbias(y_true, y_pred):
    return 100 * np.sum(y_pred - y_true) / np.sum(y_true)

## Learning rate finder development

In [ ]:
def lr_find(self, start_lr=1e-7, end_lr=10, num_iter=100, step_mode="exp", show_plot=True):
        """Find a good learning rate by training with exponentially growing lr
            source: https://github.com/fastai/fastai1/blob/master/fastai/train.py#L33

        
        Args:
            start_lr (float): Starting learning rate
            end_lr (float): Maximum learning rate
            num_iter (int): Number of iterations to run
            step_mode (str): "exp" for exponential increase, "linear" for linear increase
            show_plot (bool): Whether to display the loss plot
            
        Returns:
            tuple: (optimal_lr, learning_rates, losses)
        """
        # Save the original model state
        original_state = {
            'model': self.model.state_dict(),
            'optimizer': self.optimizer
        }
        
        # Initialize optimizer with start_lr
        optimizer = self.optimizer(self.model.parameters(), lr=start_lr)
        
        # Calculate the multiplication factor for each step
        if step_mode == "exp":
            gamma = (end_lr / start_lr) ** (1 / num_iter)
        else:
            gamma = (end_lr - start_lr) / num_iter
            
        scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma) if step_mode == "exp" else None
        
        learning_rates = []
        losses = []
        best_loss = float('inf')
        
        # Create iterator for training data
        iterator = iter(self.train_loader)
        
        for iteration in range(num_iter):
            try:
                batch_X, batch_y = next(iterator)
            except StopIteration:
                iterator = iter(self.train_loader)
                batch_X, batch_y = next(iterator)
                
            # Forward pass
            self.model.train()
            optimizer.zero_grad()
            outputs = self.model(batch_X)
            loss = self.criterion(outputs, batch_y.squeeze())
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Store the values
            current_lr = optimizer.param_groups[0]['lr']
            learning_rates.append(current_lr)
            losses.append(loss.item())
            
            # Update learning rate
            if step_mode == "exp":
                scheduler.step()
            else:
                for param_group in optimizer.param_groups:
                    param_group['lr'] = start_lr + (gamma * (iteration + 1))
            
            # Stop if the loss is exploding
            if iteration > 0 and losses[-1] > 4 * best_loss:
                break
                
            if losses[-1] < best_loss:
                best_loss = losses[-1]
        
        # Restore the original model state
        self.model.load_state_dict(original_state['model'])
        
        if show_plot:
            plt.figure(figsize=(10, 6))
            plt.plot(learning_rates, losses)
            plt.xscale('log')
            plt.xlabel('Learning Rate (log scale)')
            plt.ylabel('Loss')
            plt.title('Learning Rate Finder')
            plt.show()
            
        # Find the point of steepest descent
        smoothed_losses = np.array(losses)
        min_grad_idx = np.gradient(smoothed_losses).argmin()
        optimal_lr = learning_rates[min_grad_idx]
            
        return optimal_lr, learning_rates, losses